In [1]:
import seisbench.data as sbd
import seisbench.generate as sbg
import numpy as np
import pandas as pd
import torch
import seisbench.data as sbd
import seisbench.generate as sbg

from torch.utils.data import Dataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
import h5py

from __future__ import annotations

from dataclasses import dataclass, asdict
from typing import Dict, List, Sequence, Optional, Tuple

import numpy as np
import pandas as pd
import optuna
from geopy.distance import geodesic
from tqdm.auto import tqdm
import plotly.express as px
import json

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys

sys.path.append("PLAN/")

In [4]:
from PLAN.utils.model_ridgecrest_vision import Main_GCNN
from PLAN.utils.utils import (
    construct_dataloader,
    load_continous_data,
    load_model,
    pred,
    setup_seed,
)
from PLAN.utils.continue_dataloader import GNNDataset_ridge

setup_seed(20)

In [5]:
def export_station_file_from_stead(meta: pd.DataFrame, out_path: str) -> pd.DataFrame:
    """
    Create a PLAN-style station file from STEAD metadata.
    """
    station_df = meta[
            [
                "network_code",
                "receiver_code",
                "receiver_latitude",
                "receiver_longitude",
                "receiver_elevation_m",
            ]
        ].copy()

    station_df.columns = ["#Network", "Station", "Latitude", "Longitude", "Elevation"]
    station_df["Sitename"] = ""
    station_df["StartTime"] = "1900-01-01T00:00:00"
    station_df["EndTime"] = "3000-01-01T00:00:00"

    station_df.to_csv(out_path, sep="|", index=False)
    return station_df


def get_waveforms(trace_ids):
    waveforms = []
    with h5py.File("data/chunk4.hdf5", "r") as h5:
        for trace_id in trace_ids:
            # ENZ -> ZEN
            x = np.asarray(h5["data"][trace_id])
            x = x[:, np.argsort([2, 1, 0])]
            x = x.transpose(1, 0)
            waveforms.append(x)
        return np.stack(waveforms)

In [6]:
def make_pred(
    meta,
    model_PLAN,
    device,
    source_id,
    P_stack_value,
    S_stack_value,
    P_value,
    S_value,
    time_sample,
    station_num,
):
    meta_tmp = meta[meta.source_id == source_id].drop_duplicates(
        subset=[
            "network_code",
            "receiver_code",
            "receiver_latitude",
            "receiver_longitude",
            "receiver_elevation_m",
        ]
    )
    start_time = meta_tmp.trace_start_time.min()
    station_file_path = f"PLAN/data/stations_txt/gmap-stations_{source_id}.txt"
    station_pandas = export_station_file_from_stead(meta_tmp, station_file_path)
    inputdata = get_waveforms(meta_tmp.trace_name.to_list())

    ridge_loader, batch_start_time = construct_dataloader(
        inputdata,
        station_file_path,
        # start_time = 30000,
        # end_time = 70000,
        start_time=0,
        end_time=inputdata.shape[-1],
        interval=500,
        batchsize=1,
        num_workers=0,
    )

    return pred(
        model_PLAN,
        ridge_loader,
        station_file_path,
        device,
        batch_start_time,
        P_stack_value=P_stack_value,
        S_stack_value=S_stack_value,
        P_value=P_value,
        S_value=S_value,
        time_sample=time_sample,
        station_num=station_num,
        time_start=str(start_time),
    )


In [7]:
meta = pd.read_csv("data/chunk4.csv", sep=",")
meta["trace_start_time"] = pd.to_datetime(
    meta["trace_start_time"], format="mixed", errors="coerce"
)
# meta.trace_start_time.dt.strftime("%Y-%m-%d %H:%M:%S.%f")
meta = meta.drop_duplicates(
    subset=[
        "source_id",
        "network_code",
        "receiver_code",
        "receiver_latitude",
        "receiver_longitude",
        "receiver_elevation_m",
    ]
)
sources_cnt = meta.source_id.value_counts()
sources_cnt = sources_cnt.sample(len(sources_cnt), random_state=20)

source_ids = sources_cnt[sources_cnt > 3].index
# val_ids, test_ids = source_ids[:1000], source_ids[1000:]


with open("PLAN/Notebook/source_ids.json", "r") as f:
    d = json.load(f)
    val_ids = d["val"]
    test_ids = d["test"]

device = torch.device("cuda")
model_PLAN = Main_GCNN("Trans").to(device)
load_model_name = "PLAN/model/model_PLAN_Ridge_continue.pt"
model_PLAN = load_model(load_model_name, model_PLAN)

/tmp/ipykernel_41030/1548772038.py:1: DtypeWarning: Columns (0: source_mechanism_strike_dip_rake) have mixed types. Specify dtype option on import or set low_memory=False.
  meta = pd.read_csv("data/chunk4.csv", sep=",")


In [8]:
total_params = sum(p.numel() for p in model_PLAN.parameters())
total_params

76717948

In [ ]:
# import json

# with open("PLAN/Notebook/source_ids.json", "+w") as f:
#     d = {'val': list(val_ids), 'test': list(test_ids)}
#     json.dump(d, f)

In [8]:
P_stack_value = 1
S_stack_value = 0 
P_value = 0
S_value = 0 
time_sample = 2000
station_num = 2

In [60]:
earthquake_time_list_p, earthquake_time_list_s, earthquake_time_list_total, earthquake_loc_list = make_pred(
        meta,
        model_PLAN,
        device,
        source_ids[0],
        P_stack_value=P_stack_value,
        S_stack_value=S_stack_value,
        P_value=P_value,
        S_value=S_value,
        time_sample=time_sample,
        station_num=station_num)


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 12/12 [00:00<00:00, 13.89it/s]


# optimization

In [20]:
SOURCE_ID_COL = "source_id"
GT_TIME_COL = "source_origin_time"
GT_LAT_COL = "source_latitude"
GT_LON_COL = "source_longitude"
GT_DEPTH_COL = "source_depth_km"


@dataclass(frozen=True)
class PlanParams:
    P_stack_value: float
    S_stack_value: float
    P_value: float
    S_value: float
    time_sample: int
    station_num: int


def geodesic_km(lat1, lon1, lat2, lon2) -> float:
    try:
        return geodesic((float(lat1), float(lon1)), (float(lat2), float(lon2))).km
    except:
        return 1e6


def build_gt(meta: pd.DataFrame, source_ids: Sequence) -> pd.DataFrame:
    gt = (
        meta.loc[meta[SOURCE_ID_COL].isin(source_ids),
                 [SOURCE_ID_COL, GT_TIME_COL, GT_LAT_COL, GT_LON_COL, GT_DEPTH_COL]]
        .drop_duplicates(SOURCE_ID_COL)
        .copy()
    )
    gt[GT_TIME_COL] = pd.to_datetime(gt[GT_TIME_COL], errors="coerce")
    gt = gt.dropna(subset=[GT_TIME_COL, GT_LAT_COL, GT_LON_COL])
    return gt.rename(
        columns={
            SOURCE_ID_COL: "source_id",
            GT_TIME_COL: "gt_time",
            GT_LAT_COL: "gt_lat",
            GT_LON_COL: "gt_lon",
            GT_DEPTH_COL: "gt_depth",
        }
    ).reset_index(drop=True)


def plan_preds(meta, source_id, model_PLAN, device, params: PlanParams) -> pd.DataFrame:
    pred_out = make_pred(
        meta,
        model_PLAN,
        device,
        source_id,
        P_stack_value=params.P_stack_value,
        S_stack_value=params.S_stack_value,
        P_value=params.P_value,
        S_value=params.S_value,
        time_sample=params.time_sample,
        station_num=params.station_num,
    )

    _, _, t_all, locs = pred_out

    rows = [
        {
            "pred_time": pd.to_datetime(t, errors="coerce"),
            "pred_lat": float(loc[0]),
            "pred_lon": float(loc[1]),
            "pred_depth": float(loc[2]),
        }
        for t, loc in zip(t_all, locs)
        if t is not None and loc is not None and len(loc) >= 3
    ]
    df = pd.DataFrame(rows)
    return df.dropna(subset=["pred_time", "pred_lat", "pred_lon"]) if len(df) else df


def score_one_source(gt_row: pd.Series, pred_df: pd.DataFrame) -> Dict[str, float]:
    n_pred = len(pred_df)
    count_penalty = abs(n_pred - 1)

    if n_pred == 0:
        return {
            "count_penalty": count_penalty,
            "best_dt_sec": 1e6,
            "best_dist_km": 1e6,
            "best_depth_km": 1e6,
        }

    gt_time = gt_row["gt_time"]
    gt_lat = gt_row["gt_lat"]
    gt_lon = gt_row["gt_lon"]
    gt_depth = gt_row["gt_depth"]

    best_dt = 1e6
    best_dist = 1e6
    best_depth = 1e6

    for r in pred_df.itertuples(index=False):
        dt = abs((r.pred_time - gt_time).total_seconds())
        dist = geodesic_km(gt_lat, gt_lon, r.pred_lat, r.pred_lon)
        dep = abs(float(r.pred_depth) - float(gt_depth)) if pd.notna(gt_depth) else 0.0

        score = dt + dist + dep
        if score < best_dt + best_dist + best_depth:
            best_dt = dt
            best_dist = dist
            best_depth = dep

    return {
        "count_penalty": count_penalty,
        "best_dt_sec": best_dt,
        "best_dist_km": best_dist,
        "best_depth_km": best_depth,
    }


def evaluate_plan(meta, source_ids, model_PLAN, device, params):
    gt = build_gt(meta, source_ids).set_index("source_id")

    count_penalties = []
    dt_list, dist_list, depth_list = [], [], []

    for sid in source_ids:
        if sid not in gt.index:
            continue

        pred_df = plan_preds(meta, sid, model_PLAN, device, params)
        m = score_one_source(gt.loc[sid], pred_df)

        count_penalties.append(m["count_penalty"])
        dt_list.append(m["best_dt_sec"])
        dist_list.append(m["best_dist_km"])
        depth_list.append(m["best_depth_km"])

    return {
        "count_penalty": float(np.median(count_penalties)),
        "time_mae_sec": float(np.median(dt_list)),
        "loc_mae_km": float(np.median(dist_list)),
        "depth_mae_km": float(np.median(depth_list)),
    }, pd.DataFrame({"count_penalty": count_penalties, "dt": dt_list, "dist": dist_list, "depth": depth_list})


def plan_objective(metrics):
    return (
        1000.0 * metrics["count_penalty"]
        + min(metrics["time_mae_sec"], 30.0) / 10.0
        + min(metrics["loc_mae_km"], 500.0) / 50.0
        + min(metrics["depth_mae_km"], 100.0) / 20.0
    )


def objective(trial: optuna.Trial, meta, val_ids, model_PLAN, device) -> float:
    P_stack_value = trial.suggest_float("P_stack_value", 0.5, 2.0)
    S_stack_value = trial.suggest_float("S_stack_value", 0.0, min(P_stack_value, 0.8))

    P_value = trial.suggest_float("P_value", 0.0, 0.0)
    S_value = trial.suggest_float("S_value", 0.0,  0.0)

    params = PlanParams(
        P_stack_value=P_stack_value,
        S_stack_value=S_stack_value,
        P_value=P_value,
        S_value=S_value,
        time_sample=trial.suggest_int("time_sample", 100, 500, step=30),
        station_num=trial.suggest_int("station_num", 2, 5),
    )

    metrics, _ = evaluate_plan(meta, val_ids, model_PLAN, device, params)

    trial.set_user_attr("metrics", metrics)

    return plan_objective(metrics)


def optimize_plan(meta, val_ids, model_PLAN, device, n_trials=50, seed=20):
    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=seed),
    )
    study.enqueue_trial(
    {
        "P_stack_value": 1.0,
        "S_stack_value": 0.0,
        "P_value": 0.0,
        "S_value": 0.0,
        "time_sample": 200,
        "station_num": 2,
    }
)
    study.optimize(lambda t: objective(t, meta, val_ids, model_PLAN, device), n_trials=n_trials)

    best = study.best_trial
    best_params = PlanParams(**best.params)
    best_metrics = best.user_attrs["metrics"]
    trials_df = pd.DataFrame(
        [{**t.params, **(t.user_attrs.get("metrics", {}) if t.user_attrs else {}), "objective": t.value}
         for t in study.trials]
    )
    return best_params, best_metrics, trials_df, study

In [10]:
best_params, best_val_metrics, trials_df, study = optimize_plan(
    meta=meta,
    val_ids=val_ids[:50],
    model_PLAN=model_PLAN,
    device=device,
    n_trials=30,
    seed=20,
)

[I 2026-05-22 02:59:36,425] A new study created in memory with name: no-name-c6b3d23e-d831-4a52-9c03-7c9c684df1d8
[I 2026-05-22 03:00:16,684] Trial 0 finished with value: 8001.316791161808 and parameters: {'P_stack_value': 1.0, 'S_stack_value': 0.0, 'P_value': 0.0, 'S_value': 0.0, 'time_sample': 200, 'station_num': 2}. Best is trial 0 with value: 8001.316791161808.
[I 2026-05-22 03:00:41,545] Trial 1 finished with value: 1018.0 and parameters: {'P_stack_value': 1.3821962016159113, 'S_stack_value': 0.7181709823275344, 'P_value': 0.0, 'S_value': 0.0, 'time_sample': 460, 'station_num': 5}. Best is trial 1 with value: 1018.0.
[I 2026-05-22 03:01:04,210] Trial 2 finished with value: 1011.7041714223479 and parameters: {'P_stack_value': 0.5538343784252799, 'S_stack_value': 0.38311913031440625, 'P_value': 0.0, 'S_value': 0.0, 'time_sample': 250, 'station_num': 4}. Best is trial 2 with value: 1011.7041714223479.
[I 2026-05-22 03:01:30,008] Trial 3 finished with value: 1004.6456925887588 and par

In [22]:
meta['trace_start_time'] = meta.trace_start_time.dt.strftime('%Y-%m-%d %H:%M:%S.%f')

In [ ]:
params = PlanParams(P_stack_value=1.8516371477775202, S_stack_value=0.7399984260034951, P_value=0.0, S_value=0.0, time_sample=340, station_num=2)
test_metrics, test_evals = evaluate_plan(meta, test_ids[:500], model_PLAN, device, params)
test_evals_clean = test_evals[test_evals.apply(lambda row: all(row != 1e6), axis=1)]


PlanParams(P_stack_value=1.8516371477775202, S_stack_value=0.7399984260034951, P_value=0.0, S_value=0.0, time_sample=340, station_num=2)


NameError: name 'best_val_metrics' is not defined

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

test_evals_clean = test_evals[test_evals.apply(lambda row: all(row != 1e6), axis=1)]

fig = make_subplots(rows=2, cols=2, subplot_titles=('count_penalty', 'dt', 'dist', 'depth'))

for i, col in enumerate(['count_penalty', 'dt', 'dist', 'depth']):
    fig.add_trace(
        go.Histogram(x=test_evals_clean[col], name=col),
        row=i // 2 + 1,
        col=i % 2 + 1,
    )

fig.update_layout(
    height=1000,
    width=1000,
    title="Histograms of PLAN evaluation metrics",
)

fig.show()


NameError: name 'test_evals' is not defined

In [27]:
test_evals[test_evals.apply(lambda row: all(row != 1e6), axis=1)].mean()

count_penalty     1.937282
dt                4.848746
dist             27.865168
depth             5.596526
dtype: float64

In [ ]:
test_metrics, test_evals = evaluate_plan(meta, test_ids[:500], model_PLAN, device, best_params)
print(best_params)
print(best_val_metrics)
print(test_metrics)

PlanParams(P_stack_value=0.5538343784252799, S_stack_value=0.38311913031440625, P_value=0.0, S_value=0.0, time_sample=250, station_num=4)
{'count_penalty': 1.0, 'time_mae_sec': 20.056518500000003, 'loc_mae_km': 1000000.0, 'depth_mae_km': 8.40197361946106}
{'count_penalty': 1.0, 'time_mae_sec': 26.207955, 'loc_mae_km': 77.29830539677702, 'depth_mae_km': 8.881104850769043}
